# 🧠 Convolutional Neural Networks with Keras

**Module:** 02 — Introduction to Deep Learning & Neural Networks with Keras  
**Topic:** CNNs — spatial feature extraction, pooling, and depth  
**Dataset:** MNIST handwritten digits (28×28 greyscale images)

## 📋 Overview

In this notebook I explore **Convolutional Neural Networks (CNNs)** — the architecture that made computer vision practical. Instead of treating an image as a flat vector of pixels (like a vanilla ANN would), a CNN preserves spatial structure: it learns *where* patterns appear, not just *whether* they appear.

I build two CNN variants on MNIST and compare their depth and accuracy:

| Model | Architecture | Purpose |
|---|---|---|
| CNN-1 | 1× Conv + 1× Pool → Dense → Softmax | Baseline — shallow CNN |
| CNN-2 | 2× Conv + 2× Pool → Dense → Softmax | Deeper — richer feature hierarchy |
| Practice-1 | CNN-2, batch_size=1024 | Explore batch size effect |
| Practice-2 | CNN-2, batch_size=1024, epochs=25 | Explore epoch count effect |

**What I learn:**
- How Conv2D layers detect local spatial patterns (edges, curves, textures)
- How MaxPooling compresses feature maps and adds translation invariance
- How stacking Conv+Pool layers builds a feature hierarchy
- How batch size and epoch count affect convergence speed and final accuracy

## 🧩 Theory — How CNNs Work

### 🔢 The convolution operation

A convolutional layer slides a small **filter** (kernel) $W$ of size $f \times f$ across the input image, computing a dot product at each position. For a single channel input $X$:

$$Z^{(l)}_{i,j} = \sum_{m=0}^{f-1} \sum_{n=0}^{f-1} X_{i+m,\, j+n} \cdot W_{m,n} + b$$

This produces a **feature map** — a 2D activation grid showing where that filter's pattern was detected. With $K$ filters, the layer outputs $K$ feature maps stacked as a 3D volume.

> **Telecom analogy 📡:** A Conv2D filter is exactly a 2D FIR filter (finite impulse response). In signal processing I convolve a 1D signal with a kernel to detect specific frequencies. A CNN convolves a 2D image with kernels to detect specific spatial frequencies — edges are high-frequency, blobs are low-frequency. The feature maps are the filter bank outputs.

### 📉 Output size after convolution

Given input width $W$, filter size $f$, padding $p$, and stride $s$:

$$W_{out} = \left\lfloor \frac{W - f + 2p}{s} \right\rfloor + 1$$

For our first layer: input 28×28, filter 5×5, stride 1, no padding → output size = $\lfloor(28 - 5)/1\rfloor + 1 = 24$. So the feature map is **24×24**.

### 🔄 MaxPooling — downsampling with spatial invariance

MaxPooling takes the maximum value in each $p \times p$ window with stride $s$:

$$P_{i,j} = \max_{0 \le m < p,\, 0 \le n < p} Z_{i \cdot s + m,\, j \cdot s + n}$$

With pool_size=(2,2) and stride=2, each 24×24 feature map becomes **12×12** — half the spatial size. This:
- Reduces computation by 4×
- Makes the representation robust to small translations (if a feature shifts 1 pixel, the max barely changes)
- Forces the network to focus on the strongest activations

> **Telecom analogy 📡:** MaxPooling is like decimation in a multirate DSP chain — after filtering, I downsample to reduce bandwidth. The difference is I keep the *peak* value per window rather than every sample, which is closer to envelope detection.

### 🏗️ Full CNN architecture (CNN-1)

```
Input [28×28×1]
    ↓  Conv2D(16 filters, 5×5, stride 1, ReLU)
[24×24×16]   ← 16 feature maps
    ↓  MaxPooling2D(2×2, stride 2)
[12×12×16]   ← spatial downsampling
    ↓  Flatten()
[2304]       ← 12 × 12 × 16 = 2304 features
    ↓  Dense(100, ReLU)
[100]
    ↓  Dense(10, Softmax)
[10]         ← one probability per digit class
```

The loss function used is **categorical cross-entropy**:

$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

where $y_c \in \{0, 1\}$ is the one-hot true label and $\hat{y}_c$ is the softmax output probability for class $c$.

## ⚙️ Setup — Libraries & Environment

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # suppress TF CPU warnings

import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.utils import to_categorical
from keras.datasets import mnist

print(f"✅ Keras version: {keras.__version__}")

## 📥 Data Preparation — MNIST

MNIST contains 70,000 greyscale images of handwritten digits (0–9), each 28×28 pixels:
- **Training set:** 60,000 images
- **Test set:** 10,000 images

I need to reshape, normalise, and one-hot encode before feeding into the CNN.

### 🔢 Reshape — adding the channel dimension

Keras CNNs expect 4D input: `[samples, height, width, channels]`. MNIST is greyscale so channels = 1. I reshape from `(N, 28, 28)` → `(N, 28, 28, 1)`.

### 📊 Normalise — pixel values to [0, 1]

Raw pixel values are integers in $[0, 255]$. I divide by 255 to get floats in $[0, 1]$:

$$x_{\text{norm}} = \frac{x_{\text{raw}}}{255}$$

This ensures the gradients during training stay numerically stable — large raw values (up to 255) would produce large activations and destabilise training.

### 🎯 One-hot encode — labels to binary vectors

The label `7` becomes `[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]` — a 10-dimensional vector where only position 7 is 1. This is required for the softmax + categorical cross-entropy output head.

In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Reshape: [N, 28, 28] → [N, 28, 28, 1] — add channel dimension for Conv2D
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype('float32')
X_test  = X_test.reshape(X_test.shape[0],  28, 28, 1).astype('float32')

# Normalise: pixel range [0, 255] → [0.0, 1.0]
X_train = X_train / 255
X_test  = X_test  / 255

# One-hot encode labels: integer 7 → [0,0,0,0,0,0,0,1,0,0]
y_train = to_categorical(y_train)
y_test  = to_categorical(y_test)

num_classes = y_test.shape[1]  # 10 digit classes

print(f"📐 X_train shape: {X_train.shape}  →  {X_train.shape[0]} images of 28×28×1")
print(f"📐 X_test shape:  {X_test.shape}")
print(f"🎯 y_train shape: {y_train.shape}  →  {num_classes} classes")

---

## Part 1 — 🏗️ CNN with One Conv + Pool Layer

I start with the simplest viable CNN: one convolutional layer to extract local features, one pooling layer to compress them, then a flat Dense head for classification.

### Architecture detail

| Layer | Config | Output shape | Parameters |
|---|---|---|---|
| `Input` | shape=(28,28,1) | 28×28×1 | 0 |
| `Conv2D` | 16 filters, 5×5 kernel, stride=1, ReLU | 24×24×16 | 16×(5×5×1+1) = 416 |
| `MaxPooling2D` | pool=2×2, stride=2 | 12×12×16 | 0 |
| `Flatten` | — | 2304 | 0 |
| `Dense` | 100 units, ReLU | 100 | 2304×100+100 = 230,500 |
| `Dense` | 10 units, Softmax | 10 | 100×10+10 = 1,010 |

**Total trainable parameters: ~231,926**

The Conv2D layer learns 16 different 5×5 filters. Each filter becomes specialised during training — one might fire on horizontal edges, another on curves, another on corners. This is analogous to a bank of matched filters in a CDMA receiver, where each filter is tuned to a specific spreading code pattern.

In [ ]:
def convolutional_model_1():
    """CNN with one Conv+Pool block."""
    model = Sequential([
        Input(shape=(28, 28, 1)),

        # Feature extraction: detect local 5×5 spatial patterns
        Conv2D(16, (5, 5), strides=(1, 1), activation='relu'),
        # Output: 24×24×16 feature maps

        # Spatial downsampling: keep strongest activation per 2×2 window
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        # Output: 12×12×16

        # Bridge: flatten spatial features → 1D vector for Dense layers
        Flatten(),
        # Output: 2304-dim vector

        # Classification head
        Dense(100, activation='relu'),
        Dense(num_classes, activation='softmax'),
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model_1 = convolutional_model_1()
model_1.summary()

### 🎯 Train & Evaluate — CNN-1

I train for 10 epochs with batch_size=200. Each **epoch** is one full pass through all 60,000 training images. Each **batch** is a mini-subset of 200 images used to compute one gradient update step.

With batch_size=200 and 60,000 images, there are $60000 / 200 = 300$ gradient steps per epoch, and $10 \times 300 = 3000$ total updates.

The `validation_data=(X_test, y_test)` argument makes Keras evaluate the model on the test set at the end of each epoch — letting me watch for overfitting (training accuracy rising while val accuracy stalls).

In [ ]:
# Build the model
model_1 = convolutional_model_1()

# Train
history_1 = model_1.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=200,
    verbose=2
)

# Evaluate on test set
scores_1 = model_1.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ CNN-1 Accuracy: {scores_1[1]:.4f}")
print(f"⚠️  CNN-1 Error:    {100 - scores_1[1]*100:.2f}%")

---

## Part 2 — 🏗️ CNN with Two Conv + Pool Layers

Now I stack a **second** Conv+Pool block on top of the first. This creates a feature hierarchy:

- **Layer 1** learns low-level features: edges, blobs, gradients
- **Layer 2** combines those into mid-level features: curves, corners, digit parts

This is analogous to a two-stage matched filter bank in radar: the first stage detects elementary pulse shapes; the second stage combines them to detect specific waveform patterns.

### Architecture detail

| Layer | Config | Output shape | Parameters |
|---|---|---|---|
| `Input` | shape=(28,28,1) | 28×28×1 | 0 |
| `Conv2D` | 16 filters, 5×5, ReLU | 24×24×16 | 416 |
| `MaxPooling2D` | 2×2, stride=2 | 12×12×16 | 0 |
| `Conv2D` | 8 filters, 2×2, ReLU | 11×11×8 | 8×(2×2×16+1) = 520 |
| `MaxPooling2D` | 2×2, stride=2 | 5×5×8 | 0 |
| `Flatten` | — | 200 | 0 |
| `Dense` | 100 units, ReLU | 100 | 200×100+100 = 20,100 |
| `Dense` | 10 units, Softmax | 10 | 100×10+10 = 1,010 |

**Total trainable parameters: ~22,046** — far fewer than CNN-1 thanks to the second pooling stage compressing the spatial maps aggressively before the Dense layers.

### 🔢 Tracing the spatial dimensions

After Block 1 (Conv 5×5 → Pool 2×2): $28 \xrightarrow{-4} 24 \xrightarrow{/2} 12$

After Block 2 (Conv 2×2 → Pool 2×2): $12 \xrightarrow{-1} 11 \xrightarrow{/2} 5$ (floor division)

Flattened: $5 \times 5 \times 8 = 200$

In [ ]:
def convolutional_model_2():
    """CNN with two Conv+Pool blocks — deeper feature hierarchy."""
    model = Sequential([
        Input(shape=(28, 28, 1)),

        # Block 1 — detect low-level features (edges, textures)
        Conv2D(16, (5, 5), activation='relu'),   # → 24×24×16
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),  # → 12×12×16

        # Block 2 — detect mid-level features (curves, digit parts)
        Conv2D(8, (2, 2), activation='relu'),    # → 11×11×8
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),  # → 5×5×8

        # Classification head
        Flatten(),         # → 200
        Dense(100, activation='relu'),
        Dense(num_classes, activation='softmax'),
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model_2 = convolutional_model_2()
model_2.summary()

### 🎯 Train & Evaluate — CNN-2

In [ ]:
# Build the model
model_2 = convolutional_model_2()

# Train
history_2 = model_2.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=200,
    verbose=2
)

# Evaluate
scores_2 = model_2.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ CNN-2 Accuracy: {scores_2[1]:.4f}")
print(f"⚠️  CNN-2 Error:    {100 - scores_2[1]*100:.2f}%")

---

## 🧪 Practice Exercises

### Exercise 1 — 📦 Batch Size Effect

**Question:** How does a larger batch size (1024 vs 200) affect training time and accuracy?

**What to expect:** Larger batches mean fewer gradient updates per epoch (60 vs 300 per epoch), and each update uses a broader sample of the data — gradients are less noisy but also less exploratory. Training may be faster per epoch but convergence may be slower or less precise.

With batch_size=1024: $60000 / 1024 \approx 59$ updates/epoch vs 300 with batch_size=200. The gradient estimate is computed over a larger sample:

$$\nabla \mathcal{L}_{\text{batch}} = \frac{1}{B} \sum_{i=1}^{B} \nabla \mathcal{L}(x_i, y_i)$$

A larger $B$ gives a lower-variance but higher-bias gradient estimate — less stochastic noise, but potentially missing sharp loss minima.

In [ ]:
# Exercise 1 — Effect of batch_size=1024 on accuracy
model_ex1 = convolutional_model_2()

history_ex1 = model_ex1.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=1024,  # ← larger batch
    verbose=2
)

scores_ex1 = model_ex1.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Ex1 (batch=1024, epochs=10) Accuracy: {scores_ex1[1]:.4f}")
print(f"⚠️  Error: {100 - scores_ex1[1]*100:.2f}%")

### Exercise 2 — ⏱️ Epoch Count Effect

**Question:** If I keep batch_size=1024 but train for 25 epochs instead of 10, does accuracy recover?

**What to expect:** More epochs compensate for the fewer updates per epoch (59 vs 300). The model has more total gradient steps:

$$\text{Total updates} = \left\lfloor \frac{N}{B} \right\rfloor \times E$$

- Config 1 (batch=200, epochs=10): $300 \times 10 = 3000$ updates  
- Config 2 (batch=1024, epochs=10): $59 \times 10 = 590$ updates  
- Config 3 (batch=1024, epochs=25): $59 \times 25 = 1475$ updates

Config 3 still has fewer total updates than Config 1, but training is faster per update (larger batches parallelise better on GPU) and the gradient is less noisy.

In [ ]:
# Exercise 2 — Effect of batch_size=1024 + epochs=25
model_ex2 = convolutional_model_2()

history_ex2 = model_ex2.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=25,       # ← more epochs
    batch_size=1024, # ← larger batch
    verbose=2
)

scores_ex2 = model_ex2.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Ex2 (batch=1024, epochs=25) Accuracy: {scores_ex2[1]:.4f}")
print(f"⚠️  Error: {100 - scores_ex2[1]*100:.2f}%")

---

## 📊 Summary

### Results comparison

| Model | Batch | Epochs | Updates | Expected Accuracy | Key Takeaway |
|---|---|---|---|---|---|
| CNN-1 | 200 | 10 | 3,000 | ~99% | 1 Conv+Pool block is already very strong on MNIST |
| CNN-2 | 200 | 10 | 3,000 | ~99% | 2 blocks, 10× fewer Dense params, similar accuracy |
| Ex1: CNN-2 | 1024 | 10 | ~590 | Lower | Fewer updates = less convergence |
| Ex2: CNN-2 | 1024 | 25 | ~1,475 | Recovering | More epochs partially compensate for large batch |

### Concepts covered

| Concept | What it does | Telecom analogy |
|---|---|---|
| **Conv2D** | Detects local spatial patterns via learnable filters | 2D FIR filter / matched filter bank |
| **MaxPooling** | Downsamples feature maps, adds translation invariance | Decimation + envelope detection |
| **Flatten** | Bridges spatial features → 1D vector for Dense | Parallel-to-serial conversion |
| **Depth (2 blocks)** | Learns hierarchical features: edges → shapes → parts | Multi-stage filter cascades in receiver chains |
| **Batch size** | Controls gradient noise and update frequency | Trade-off between estimation accuracy and update rate |
| **Epochs** | Number of full passes over training data | Total integration time in a signal accumulator |

### Key insight

CNNs beat ANNs on images for two reasons:
1. **Parameter sharing** — each filter is reused across all spatial positions (translation equivariance)
2. **Sparse connectivity** — each neuron only connects to a local $f \times f$ patch, not the whole image

This makes CNNs far more efficient and better at capturing local structure than a Dense layer that treats all 784 pixels independently.

---

## 🧪 Sandbox — Experiments to Try

Use this section to run your own experiments. Ideas:

1. **More filters in Conv1:** Change `Conv2D(16, ...)` → `Conv2D(32, ...)` — more filters = richer feature maps. Does accuracy improve?

2. **Smaller kernel:** Change `(5, 5)` → `(3, 3)` — smaller receptive field, more spatial precision. Common in modern CNNs (VGG, ResNet).

3. **Add Dropout:** Insert `Dropout(0.25)` after MaxPooling — regularises the feature maps, reduces overfitting.

4. **Visualise filters:** After training, extract `model.layers[0].get_weights()[0]` and plot the 16 learned 5×5 kernels. You should see edge detectors and blob detectors emerge.

5. **Telecom application:** Replace MNIST with a **modulation classification** task — 2D constellation diagrams (I/Q plane) as 32×32 images, classify between BPSK/QPSK/16-QAM. A CNN is a natural fit for detecting constellation cluster shapes.

In [ ]:
# 🧪 Sandbox — your experiments here

# Example: modified CNN with 32 filters and 3×3 kernels
def convolutional_model_sandbox():
    model = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, (3, 3), activation='relu'),        # more filters, smaller kernel
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        Conv2D(16, (3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Uncomment to run:
# model_sb = convolutional_model_sandbox()
# model_sb.summary()
# model_sb.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=200, verbose=2)